# KITTI Depth Completion: build the ready-to-use Train / Val / Test dataset

Input: the 61 zips in `MyDrive/KITTI_DC/zips/` (from `kitti_download_colab.ipynb`).

Output, every pair cropped to **1216 × 352**, the three files of one pair share the **same file name**:
```
/content/kitti/
├─ train/  rgb/  lidar/  gt/     8,237 pairs
├─ val/    rgb/  lidar/  gt/     1,023 pairs
└─ test/   rgb/  lidar/  gt/     1,000 pairs
```
File name: `<drive>_<frame>_<camera>.png`, e.g. `2011_09_26_drive_0017_0000000005_image_02.png`.

| Folder | Content | Format |
| :--- | :--- | :--- |
| `rgb/` | camera image (input A) | 8-bit RGB PNG |
| `lidar/` | sparse LiDAR depth (input B) | 16-bit PNG, metres = value / 256, 0 = no point |
| `gt/` | ground-truth depth (target) | 16-bit PNG, metres = value / 256, 0 = no GT |

**How to use**
- **Part A: run once.** Builds the dataset and saves it to Drive as `KITTI_DC/dataset/{train,val,test}.tar` (about 10 GB).
- **Part B: run at the start of every session.** Copies the 3 tar files from Drive to the fast Colab disk.
- **Part C:** PyTorch `Dataset` + a check that everything loads.

# Part A: build the dataset (run once)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, io, time, subprocess, zipfile, multiprocessing
from concurrent.futures import ProcessPoolExecutor
from PIL import Image

ZIPS = '/content/drive/MyDrive/KITTI_DC/zips'      # the 61 downloaded zips
DST  = '/content/drive/MyDrive/KITTI_DC/dataset'   # train.tar / val.tar / test.tar are saved here
OUT  = '/content/kitti'                            # dataset is built on the fast Colab disk
H, W = 352, 1216
SPLITS, KINDS = ('train', 'val', 'test'), ('rgb', 'lidar', 'gt')

TRAIN = [
    '2011_09_26_drive_0017',
    '2011_09_26_drive_0027',
    '2011_09_26_drive_0032',
    '2011_09_26_drive_0039',
    '2011_09_26_drive_0056',
    '2011_09_26_drive_0086',
    '2011_09_26_drive_0101',
    '2011_09_28_drive_0001',
    '2011_09_28_drive_0045',
    '2011_09_28_drive_0054',
    '2011_09_28_drive_0068',
    '2011_09_28_drive_0070',
    '2011_09_28_drive_0078',
    '2011_09_28_drive_0090',
    '2011_09_28_drive_0094',
    '2011_09_28_drive_0095',
    '2011_09_28_drive_0096',
    '2011_09_28_drive_0098',
    '2011_09_28_drive_0100',
    '2011_09_28_drive_0104',
    '2011_09_28_drive_0113',
    '2011_09_28_drive_0117',
    '2011_09_28_drive_0126',
    '2011_09_28_drive_0135',
    '2011_09_28_drive_0138',
    '2011_09_28_drive_0143',
    '2011_09_28_drive_0145',
    '2011_09_28_drive_0155',
    '2011_09_28_drive_0165',
    '2011_09_28_drive_0167',
    '2011_09_28_drive_0183',
    '2011_09_28_drive_0184',
    '2011_09_28_drive_0192',
    '2011_09_28_drive_0195',
    '2011_09_28_drive_0199',
    '2011_09_28_drive_0204',
    '2011_09_28_drive_0205',
    '2011_09_28_drive_0214',
    '2011_09_28_drive_0216',
    '2011_09_28_drive_0220',
    '2011_09_30_drive_0027',
    '2011_09_30_drive_0033',
    '2011_10_03_drive_0042',
]

VAL = [
    '2011_09_26_drive_0001',
    '2011_09_26_drive_0011',
    '2011_09_26_drive_0018',
    '2011_09_26_drive_0048',
    '2011_09_26_drive_0052',
    '2011_09_26_drive_0060',
    '2011_09_28_drive_0057',
    '2011_09_28_drive_0065',
    '2011_09_28_drive_0066',
    '2011_09_28_drive_0089',
    '2011_09_28_drive_0102',
    '2011_09_28_drive_0122',
    '2011_09_28_drive_0136',
    '2011_09_28_drive_0156',
    '2011_09_28_drive_0208',
]

### A1. Find every complete pair inside the zips
- Train / Val: RGB from `<drive>_sync.zip`, LiDAR from `data_depth_velodyne.zip`, GT from `data_depth_annotated.zip`, camera `image_02`.
- Test: RGB, LiDAR and GT from `data_depth_selection.zip` → `val_selection_cropped` (both cameras).

In [ ]:
def names(zip_name):
    with zipfile.ZipFile(f'{ZIPS}/{zip_name}') as z:
        return set(z.namelist())

def frames(all_names, prefix):
    return {n[len(prefix):-4] for n in all_names if n.startswith(prefix) and n.endswith('.png')}

gt_names, lid_names, sel_names = names('data_depth_annotated.zip'), names('data_depth_velodyne.zip'), names('data_depth_selection.zip')
JOBS = {s: [] for s in SPLITS}      # job = (output name, (zip, rgb member), (zip, lidar member), (zip, gt member))

for split, drives in (('train', TRAIN), ('val', VAL)):
    skipped = 0
    for d in drives:
        p_rgb = f'{d[:10]}/{d}_sync/image_02/data/'
        p_lid = f'train/{d}_sync/proj_depth/velodyne_raw/image_02/'
        p_gt  = f'train/{d}_sync/proj_depth/groundtruth/image_02/'
        g = frames(gt_names, p_gt)
        ok = g & frames(lid_names, p_lid) & frames(names(f'{d}_sync.zip'), p_rgb)
        skipped += len(g - ok)
        for f in sorted(ok):
            JOBS[split].append((f'{d}_{f}_image_02.png',
                                (f'{d}_sync.zip', f'{p_rgb}{f}.png'),
                                ('data_depth_velodyne.zip', f'{p_lid}{f}.png'),
                                ('data_depth_annotated.zip', f'{p_gt}{f}.png')))
    print(f'{split}: {len(JOBS[split])} pairs, {skipped} GT frames without RGB or LiDAR')

S, skipped = 'depth_selection/val_selection_cropped/', 0
for n in sorted(sel_names):
    if n.startswith(S + 'image/') and n.endswith('.png'):
        f = n[len(S + 'image/'):]                    # <drive>_sync_image_<frame>_image_0X.png
        lid = S + 'velodyne_raw/' + f.replace('_sync_image_', '_sync_velodyne_raw_', 1)
        gt  = S + 'groundtruth_depth/' + f.replace('_sync_image_', '_sync_groundtruth_depth_', 1)
        if lid in sel_names and gt in sel_names:
            d, rest = f.split('_sync_image_')
            JOBS['test'].append((f'{d}_{rest}', ('data_depth_selection.zip', n),
                                 ('data_depth_selection.zip', lid), ('data_depth_selection.zip', gt)))
        else:
            skipped += 1
print(f'test: {len(JOBS["test"])} pairs, {skipped} incomplete')
print('expected: train 8237, val 1023, test 1000')

### A2. Extract, crop to 1216 × 352 and save
Uses all CPU cores. Re-running skips pairs that are already done.

In [ ]:
_zips = {}                                  # each worker process opens its own zip handles
def _zip(name):
    if name not in _zips:
        _zips[name] = zipfile.ZipFile(f'{ZIPS}/{name}')
    return _zips[name]

def crop(img):                              # bottom-center crop (the top of the image has no LiDAR)
    w, h = img.size
    l, t = (w - W) // 2, h - H
    return img.crop((l, t, l + W, t + H))

def make(job):
    split, (name, *parts) = job
    outs = [f'{OUT}/{split}/{k}/{name}' for k in KINDS]
    if all(os.path.exists(o) for o in outs):
        return
    imgs = [Image.open(io.BytesIO(_zip(z).read(m))) for z, m in parts]
    assert len({im.size for im in imgs}) == 1, f'RGB / LiDAR / GT size differ: {name}'
    for im, o in zip(imgs, outs):
        crop(im).save(o + '.tmp', format='PNG')
        os.replace(o + '.tmp', o)           # never leave a half-written file

for s in SPLITS:
    for k in KINDS:
        os.makedirs(f'{OUT}/{s}/{k}', exist_ok=True)

jobs = [(s, j) for s in SPLITS for j in JOBS[s]]
t0 = time.time()
with ProcessPoolExecutor(os.cpu_count(), mp_context=multiprocessing.get_context('fork')) as ex:
    for i, _ in enumerate(ex.map(make, jobs, chunksize=16), 1):
        if i % 1000 == 0 or i == len(jobs):
            print(f'{i}/{len(jobs)} pairs  {time.time() - t0:.0f} s')

### A3. Check the result

In [ ]:
for s in SPLITS:
    files = [sorted(f for f in os.listdir(f'{OUT}/{s}/{k}') if f.endswith('.png')) for k in KINDS]
    assert files[0] == files[1] == files[2], f'{s}: rgb / lidar / gt names differ'
    first = files[0][0]
    sizes = [Image.open(f'{OUT}/{s}/{k}/{first}').size for k in KINDS]
    modes = [Image.open(f'{OUT}/{s}/{k}/{first}').mode for k in KINDS]
    assert all(sz == (W, H) for sz in sizes), sizes
    print(f'{s:5s}: {len(files[0])} pairs, size {sizes[0]}, modes {modes}, example {first}')

### A4. Save to Google Drive
One tar file per split (few big files upload much faster than thousands of small ones).
Wait until it prints `saved`; do not close the tab before that.

In [ ]:
os.makedirs(DST, exist_ok=True)
for s in SPLITS:
    subprocess.run(['tar', '-cf', f'{DST}/{s}.tar', '-C', OUT, s], check=True)
    print(f'{s}.tar  {os.path.getsize(f"{DST}/{s}.tar") / 1e9:.2f} GB')
drive.flush_and_unmount()                   # wait until the upload to Drive is finished
print('saved to', DST)

# Part B: load the dataset (run at the start of every session)
Skips the copy if `/content/kitti` is already there (for example right after Part A).

In [ ]:
import os, subprocess
OUT = '/content/kitti'
DST = '/content/drive/MyDrive/KITTI_DC/dataset'

if not all(os.path.isdir(f'{OUT}/{s}') for s in ('train', 'val', 'test')):
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(OUT, exist_ok=True)
    for s in ('train', 'val', 'test'):
        subprocess.run(['tar', '-xf', f'{DST}/{s}.tar', '-C', OUT], check=True)

for s in ('train', 'val', 'test'):
    print(s, len(os.listdir(f'{OUT}/{s}/rgb')), 'pairs')

# Part C: PyTorch Dataset
Returns `rgb` (3 × 352 × 1216, ImageNet-normalized), `lidar` (2 × 352 × 1216: depth / 80 and validity mask), `gt` (1 × 352 × 1216, metres, 0 = no GT).

In [ ]:
import os, numpy as np, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader

MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD  = np.array([0.229, 0.224, 0.225], np.float32)

class KittiDC(Dataset):
    def __init__(self, split, root='/content/kitti'):
        self.dir = f'{root}/{split}'
        self.names = sorted(os.listdir(f'{self.dir}/rgb'))
    def __len__(self):
        return len(self.names)
    def __getitem__(self, i):
        n = self.names[i]
        rgb = np.asarray(Image.open(f'{self.dir}/rgb/{n}').convert('RGB'), np.float32) / 255.0
        lid = np.asarray(Image.open(f'{self.dir}/lidar/{n}'), np.float32) / 256.0
        gt  = np.asarray(Image.open(f'{self.dir}/gt/{n}'), np.float32) / 256.0
        rgb   = torch.from_numpy(((rgb - MEAN) / STD).transpose(2, 0, 1).copy())
        lidar = torch.from_numpy(np.stack([lid / 80.0, (lid > 0).astype(np.float32)]))
        gt    = torch.from_numpy(gt[None].copy())
        return rgb, lidar, gt

train_ds, val_ds, test_ds = KittiDC('train'), KittiDC('val'), KittiDC('test')
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=os.cpu_count(), pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=os.cpu_count(), pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=8, shuffle=False, num_workers=os.cpu_count(), pin_memory=True)

rgb, lidar, gt = next(iter(train_dl))
print('sizes  :', len(train_ds), len(val_ds), len(test_ds))
print('batch  :', tuple(rgb.shape), tuple(lidar.shape), tuple(gt.shape))
print('LiDAR valid %.1f%%, GT valid %.1f%%, GT max %.1f m' % (100 * lidar[:, 1].mean(), 100 * (gt > 0).float().mean(), gt.max()))

### One example per split

In [ ]:
import matplotlib.pyplot as plt
CMAP = plt.get_cmap('turbo').copy(); CMAP.set_bad('black')   # pixels without GT are black
fig, ax = plt.subplots(3, 3, figsize=(18, 6.5))
for r, (name, ds) in enumerate([('train', train_ds), ('val', val_ds), ('test', test_ds)]):
    rgb, lidar, gt = ds[0]
    img = (rgb.numpy().transpose(1, 2, 0) * STD + MEAN).clip(0, 1)
    ys, xs = np.nonzero(lidar[1].numpy())
    ax[r, 0].imshow(img); ax[r, 0].set_title(f'{name}: RGB  {ds.names[0]}')
    ax[r, 1].imshow(img); ax[r, 1].scatter(xs, ys, c=lidar[0].numpy()[ys, xs] * 80, s=0.4, cmap='turbo', vmin=0, vmax=80)
    ax[r, 1].set_title('sparse LiDAR on RGB')
    g = gt[0].numpy()
    ax[r, 2].imshow(np.ma.masked_where(g <= 0, g), cmap=CMAP, vmin=0, vmax=80)
    ax[r, 2].set_title('GT depth (0-80 m)')
for a in ax.flat:
    a.axis('off')
plt.tight_layout(); plt.show()